# 12b — Inter-Rater Agreement & System Kappa Analysis
### Runs locally or on Colab — **no GPU required**

**Prerequisites:** place all of the following in `ANNOTATION_DIR`:

```
model1_sid_ali_assoul.csv        (filled by annotator)
model1_nadir_mahammed.csv
model1_nabil_keskes.csv
model1_houda_debza.csv
model1_master.csv                (from notebook 12a — do not edit)
... (same pattern for models 2, 3, 4)
```

**Kappa scores computed:**

| Metric | Description |
|--------|-------------|
| Pairwise Cohen's κ (6 pairs) | Agreement between each pair of the 4 annotators |
| Fleiss' κ | Overall agreement across all 4 annotators jointly |
| κ Human majority vs. Gemini Pro | Do annotators agree with Gemini's labelling? |
| κ Human majority vs. BERT | Do annotators agree with the BERT classifier? |
| κ Gemini Pro vs. BERT | Agreement between the two automated systems (subset) |

The full-test Gemini-Pro vs. BERT κ is read from the master CSV (computed in notebook 12a on the full ~5k–17k test partition).

**Tie-breaking policy (4 annotators, 2-2 tie):** the smallest label in the valid set wins. Tie counts are reported per model.

## 1. Install Dependencies

In [ ]:
!pip install -q statsmodels scikit-learn pandas numpy

## 2. Imports

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
from collections import Counter
from sklearn import preprocessing
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

print('Environment ready.')

## 3. Constants

Set `ANNOTATION_DIR` to the folder that contains the completed annotation CSVs and master files.

In [ ]:
ANNOTATION_DIR = 'annotation_sheets'  # relative to this notebook's location
                                       # full path: notebooks/v3/annotation_sheets/

ANNOTATORS = [
    'sid_ali_assoul',
    'nadir_mahammed',
    'nabil_keskes',
    'houda_debza',
]

MODEL_LABEL_SETS = {
    1: ['0', '1', '2'],
    2: ['0.0', '0.1'],
    3: ['1.0', '1.1', '1.2', '1.3'],
    4: ['2.0', '2.1', '2.2', '2.3', '2.4'],
}

MODEL_NAMES = {
    1: 'Model 1 - Overall Move Classifier (3 classes)',
    2: 'Model 2 - Move 0 Sub-move Classifier (2 classes)',
    3: 'Model 3 - Move 1 Sub-move Classifier (4 classes)',
    4: 'Model 4 - Move 2 Sub-move Classifier (5 classes)',
}

print(f'Annotators : {ANNOTATORS}')
print(f'Looking for annotation files in: {os.path.abspath(ANNOTATION_DIR)}')

## 4. Load Completed Annotation Sheets

This function:
1. Loads the master reference CSV (from notebook 12a) which contains `gemini_label` and `bert_pred_label`.
2. Loads each annotator's filled CSV (`model{N}_{name}.csv`) and merges by `id`.
3. **Validates** every annotated label — raises `ValueError` on any blank, missing, or unrecognised value.

In [ ]:
def load_completed_annotations(model_id, annotation_dir=ANNOTATION_DIR):
    """
    Loads completed annotation sheets from per-annotator subfolders:
        annotation_sheets/{annotator_name}/model{N}_{annotator_name}.csv

    Master reference is at:
        annotation_sheets/model{N}_master.csv

    Raises ValueError/FileNotFoundError if any sheet is missing or contains invalid labels.
    """
    label_set = set(MODEL_LABEL_SETS[model_id])

    master = pd.read_csv(
        f'{annotation_dir}/model{model_id}_master.csv',
        usecols=['id', 'sentence', 'gemini_label', 'bert_pred_label'],
    )

    for name in ANNOTATORS:
        path = f'{annotation_dir}/{name}/model{model_id}_{name}.csv'
        if not os.path.exists(path):
            raise FileNotFoundError(f'Missing file: {path}')

        ann_df = pd.read_csv(path)[['id', 'your_label']].copy()
        ann_df['your_label'] = ann_df['your_label'].astype(str).str.strip()

        n_blank = ann_df['your_label'].isin(['', 'nan']).sum()
        if n_blank > 0:
            raise ValueError(
                f'Model {model_id} / {name}: {n_blank} blank row(s). '
                f'All sentences must be labelled before computing kappa.'
            )

        bad = set(ann_df['your_label'].unique()) - label_set
        if bad:
            raise ValueError(
                f'Model {model_id} / {name} has invalid labels: {sorted(bad)}\n'
                f'Valid labels are: {sorted(label_set)}'
            )

        ann_df = ann_df.rename(columns={'your_label': name})
        master = master.merge(ann_df, on='id', how='left')

    print(f'Model {model_id}: loaded {len(master)} annotated sentences OK.')
    return master


ann_m1 = load_completed_annotations(1)
ann_m2 = load_completed_annotations(2)
ann_m3 = load_completed_annotations(3)
ann_m4 = load_completed_annotations(4)

## 5. Kappa Computation

**Why integer codes?**  
All kappas are computed on a shared integer code space derived from a `LabelEncoder` fit on the canonical label set. This avoids silent mismatches from CSV round-tripping — for example `'0'`, `'0.0'`, and `float(0.0)` would compare as unequal as strings, but all correctly map to integer `0` through the encoder.

In [ ]:
def majority_vote_and_ties(df, annotator_cols, label_set):
    """
    Returns (majority_vote_array, n_ties).
    Ties are broken by the smallest label in label_set.
    """
    tiebreak = label_set[0]
    votes, n_ties = [], 0

    for _, row in df.iterrows():
        labels     = [str(row[c]).strip() for c in annotator_cols]
        counts     = Counter(labels)
        max_count  = max(counts.values())
        candidates = [lbl for lbl, cnt in counts.items() if cnt == max_count]
        if len(candidates) > 1:
            n_ties += 1
            votes.append(tiebreak)
        else:
            votes.append(candidates[0])

    return np.array(votes), n_ties


def kappa_strength(k):
    if k < 0:      return 'Less than chance'
    elif k < 0.20: return 'Slight'
    elif k < 0.40: return 'Fair'
    elif k < 0.60: return 'Moderate'
    elif k < 0.80: return 'Substantial'
    else:          return 'Almost perfect'


def compute_kappas(ann_df, model_id, verbose=True):
    """
    Compute all kappa scores for one model.
    ann_df must have: sentence, gemini_label, bert_pred_label, + one column per annotator.
    Returns dict of {metric_name: value}.
    """
    label_set = MODEL_LABEL_SETS[model_id]
    le        = preprocessing.LabelEncoder().fit(label_set)
    df        = ann_df.copy()

    all_cols = ANNOTATORS + ['gemini_label', 'bert_pred_label']
    for col in all_cols:
        df[col] = df[col].astype(str).str.strip()

    nan_mask  = df[all_cols].isin(['nan', '']).any(axis=1)
    n_dropped = nan_mask.sum()
    if n_dropped > 0:
        print(f'  [WARNING] Dropped {n_dropped} row(s) with missing values.')
    df = df[~nan_mask].reset_index(drop=True)
    n  = len(df)

    for col in all_cols:
        bad = set(df[col].unique()) - set(label_set)
        if bad:
            raise ValueError(f'Column {col!r} contains labels not in label_set: {bad}')

    enc       = {col: le.transform(df[col]) for col in all_cols}
    int_range = list(range(len(label_set)))

    results = {'n_sentences': n}

    # 1. Pairwise Cohen's kappa between annotators (6 pairs)
    for a, b in itertools.combinations(ANNOTATORS, 2):
        k = cohen_kappa_score(enc[a], enc[b], labels=int_range)
        results[f'k {a} vs {b}'] = k

    # 2. Fleiss' kappa (all 4 annotators jointly)
    ratings_matrix = np.column_stack([enc[a] for a in ANNOTATORS])
    agg, _         = aggregate_raters(ratings_matrix, n_cat=len(label_set))
    results["Fleiss' k (all 4 annotators)"] = fleiss_kappa(agg)

    # 3. Majority vote + tie count
    mv_str, n_ties = majority_vote_and_ties(df, ANNOTATORS, label_set)
    mv_enc         = le.transform(mv_str)
    results['n_ties'] = n_ties

    # 4. Three system-pair kappas
    g_enc = enc['gemini_label']
    b_enc = enc['bert_pred_label']
    results['k Human majority vs Gemini Pro (subset)'] = cohen_kappa_score(mv_enc, g_enc, labels=int_range)
    results['k Human majority vs BERT (subset)']       = cohen_kappa_score(mv_enc, b_enc, labels=int_range)
    results['k Gemini Pro vs BERT (subset)']           = cohen_kappa_score(g_enc,  b_enc, labels=int_range)

    if verbose:
        print(f'\n{MODEL_NAMES[model_id]}')
        print(f'  n = {n} sentences  |  ties = {n_ties}')
        print('  ' + '-' * 70)
        for key, val in results.items():
            if key in ('n_sentences', 'n_ties'):
                continue
            print(f'  {key:<55s}  k = {val:+.4f}  [{kappa_strength(val)}]')

    return results

## 6. Dry-Run Validation

Validate the pipeline on synthetic data before trusting real results.

- **Scenario A**: annotators copy the Gemini label with 5% noise → expect κ ≈ 0.9.
- **Scenario B**: fully random labels → expect κ near 0.
- **Guard test**: an invalid label must raise `ValueError`.

In [ ]:
def make_synthetic_annotations(sample_df, label_set, noise_prob=0.05, seed=42):
    rng   = np.random.default_rng(seed)
    df    = sample_df[['id', 'sentence', 'gemini_label', 'bert_pred_label']].copy()
    n     = len(df)
    other = {lbl: [l for l in label_set if l != lbl] for lbl in label_set}

    def noisy_copy(col, prob):
        out = []
        for v in df[col]:
            sv = str(v)
            if rng.random() < prob and other.get(sv):
                out.append(rng.choice(other[sv]))
            else:
                out.append(sv)
        return out

    # High-agreement annotators (copy Gemini + small noise)
    df[ANNOTATORS[0]] = noisy_copy('gemini_label', noise_prob)
    df[ANNOTATORS[1]] = noisy_copy('gemini_label', noise_prob)
    # Low-agreement annotators (random)
    df[ANNOTATORS[2]] = [rng.choice(label_set) for _ in range(n)]
    df[ANNOTATORS[3]] = [rng.choice(label_set) for _ in range(n)]
    return df


print('=== Dry-run on synthetic annotations ===\n')
for mid, ann_df in [(1, ann_m1), (2, ann_m2), (3, ann_m3), (4, ann_m4)]:
    ls      = MODEL_LABEL_SETS[mid]
    synth_a = make_synthetic_annotations(ann_df, ls, noise_prob=0.05)
    ka      = compute_kappas(synth_a, mid, verbose=False)
    pairs_a = [v for k, v in ka.items() if ' vs ' in k]

    synth_b = make_synthetic_annotations(ann_df, ls, noise_prob=1.0)
    kb      = compute_kappas(synth_b, mid, verbose=False)
    pairs_b = [v for k, v in kb.items() if ' vs ' in k]

    print(f'  Model {mid}:  near-perfect avg k = {np.mean(pairs_a):+.3f}  |  '
          f'random avg k = {np.mean(pairs_b):+.3f}  '
          f'(ties A={ka["n_ties"]}  B={kb["n_ties"]})')

print()
try:
    bad_df = make_synthetic_annotations(ann_m1, MODEL_LABEL_SETS[1], noise_prob=0)
    bad_df.loc[0, ANNOTATORS[0]] = 'INVALID'
    compute_kappas(bad_df, 1, verbose=False)
    print('ERROR: should have raised ValueError!')
except ValueError as e:
    print(f'Invalid-label guard OK: {e}')

print('\nDry-run passed.')

## 7. Compute Real Kappa Scores

In [ ]:
all_kappas = {}
for mid, ann_df in [(1, ann_m1), (2, ann_m2), (3, ann_m3), (4, ann_m4)]:
    all_kappas[mid] = compute_kappas(ann_df, mid, verbose=True)

## 8. Full-Test Gemini-Pro vs. BERT Kappa

Computed in notebook 12a on the full test partition and stored in the master CSV.

In [ ]:
print('Full-test Gemini-Pro vs. BERT kappa (from notebook 12a):')
print('=' * 60)
for mid in [1, 2, 3, 4]:
    master = pd.read_csv(f'{ANNOTATION_DIR}/model{mid}_master.csv')
    k_full = master['full_test_gemini_vs_bert_kappa'].iloc[0]
    print(f'  {MODEL_NAMES[mid]:<55s}  k={k_full:+.4f}  [{kappa_strength(k_full)}]')

## 9. Summary Table

In [ ]:
rows = []
for mid, results in all_kappas.items():
    master = pd.read_csv(f'{ANNOTATION_DIR}/model{mid}_master.csv')
    k_full = float(master['full_test_gemini_vs_bert_kappa'].iloc[0])

    for metric, val in results.items():
        if metric in ('n_sentences', 'n_ties'):
            continue
        rows.append({
            'Model':          MODEL_NAMES[mid],
            'n':              results['n_sentences'],
            'Ties':           results['n_ties'],
            'Metric':         metric,
            'kappa':          round(val, 4),
            'Interpretation': kappa_strength(val),
        })

    rows.append({
        'Model':          MODEL_NAMES[mid],
        'n':              'full test',
        'Ties':           '-',
        'Metric':         'k Gemini Pro vs BERT (full test)',
        'kappa':          round(k_full, 4),
        'Interpretation': kappa_strength(k_full),
    })

summary_df = pd.DataFrame(rows)
display(summary_df)

In [ ]:
summary_df.to_csv('kappa_results_summary.csv', index=False)
print('Saved: kappa_results_summary.csv')

# If running on Colab:
# from google.colab import files
# files.download('kappa_results_summary.csv')